<a href="https://colab.research.google.com/github/akbarruziev660-wq/Lesson/blob/main/%D0%9F%D1%80%D0%BE%D0%B5%D0%BA%D1%82_17_18_%D0%A0%D0%B0%D0%B4%D0%B0%D1%80_%D0%BE%D1%82%D0%B7%D1%8B%D0%B2%D0%BE%D0%B2_%D0%A0%D0%95%D0%A8%D0%95%D0%9D%D0%98%D0%95150826.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛰️ Проект: «Радар отзывов»
### Финал фазы 4 · уроки 17–18 · пишем сами, получаем работающий сайт

**Задача из жизни.** Магазин получает тысячи отзывов в день. Прочитать всё вручную — недели. Сегодня ты построишь **умный анализатор тональности** и в конце — **веб-приложение с публичной ссылкой**, которое можно открыть на телефоне и показать друзьям и родителям.

**Что соберёшь по шагам:**
1. свои данные — отзывы;
2. текст → числа (токены) — мостик к уроку 17;
3. подключишь готовую модель тональности (урок 18);
4. разберёшь пачку отзывов и посмотришь на уверенность модели;
5. поймаешь «хитрые» отзывы, где модель может ошибиться;
6. нарисуешь диаграмму позитив/негатив;
7. ⭐ завернёшь всё в веб-приложение (Gradio) с публичной ссылкой.

> Запускай в Google Colab: **Среда выполнения → Выполнить все**. Ячейки выполняй сверху вниз.

## Шаг 1 · Твои данные
Хороший анализ начинается со своих данных. Собери отзывы о том, что тебе близко.

In [ ]:
# Шаг 1. Твои данные.
отзывы = [
    'Эта игра затягивает на часы, обожаю её!',
    'Ужасный сервис, больше не приду.',
    'Неплохо, но дороговато.',
    'Лучшее кафе в городе, всем советую!',
    'Ждал доставку неделю, кошмар.',
    'Ну очень «порадовал» этот магазин...',
    'Фильм неплохой, но затянут.',
    'Обожаю этот трек, слушаю на репите!',
]
print('Всего отзывов:', len(отзывы))

> ✅ **Проверка.** В выводе — «Всего отзывов: 8» (или сколько ты вписал).

## Шаг 2 · Текст → числа
Модель не видит букв. Как в уроке 17: сначала текст режется на токены-числа.

In [ ]:
# Шаг 2. Текст -> числа (как в уроке 17).
!pip install tiktoken -q
import tiktoken
enc = tiktoken.get_encoding('cl100k_base')

пример = отзывы[0]
числа = enc.encode(пример)
куски = [enc.decode([t]) for t in числа]
print('Отзыв :', пример)
print('Токены:', куски)

> ✅ **Проверка.** Ты видишь, что один отзыв разбит на кусочки-токены — некоторые короче слова.

## Шаг 3 · Готовая модель
Одна строка — и у тебя модель тональности. Ничего обучать не надо (урок 18).

In [ ]:
# Шаг 3. Подключаем готовую модель тональности (Hugging Face, без ключа).
!pip install transformers -q
from transformers import pipeline

модель = pipeline('sentiment-analysis',
                  model='nlptown/bert-base-multilingual-uncased-sentiment')

print(модель('Отличный сервис!'))

> ✅ **Проверка.** Вывод вроде `[{'label': '5 stars', 'score': ...}]`. Модель загрузилась.

🔶 **Усложни (для сильных):** добавь вторую, англоязычную модель и сравни ответы на один русский отзыв.

## Шаг 4 · Понятный вердикт
Модель отвечает «звёздами». Превратим их в ПОЗИТИВ / НЕГАТИВ / НЕЙТРАЛ + уверенность.

In [ ]:
# Шаг 4. Превращаем ответ модели в понятный вердикт.
def разобрать(отзыв):
    р = модель(отзыв)[0]
    звёзд = int(р['label'][0])
    увер  = р['score']
    вердикт = 'ПОЗИТИВ' if звёзд>=4 else ('НЕГАТИВ' if звёзд<=2 else 'НЕЙТРАЛ')
    return звёзд, вердикт, увер

print(разобрать('Лучшее кафе в городе!'))

> ✅ **Проверка.** Для «Лучшее кафе в городе!» вердикт — ПОЗИТИВ.

## Шаг 5 · Пачка отзывов
Прогоняем все свои отзывы и считаем, чего больше.

In [ ]:
# Шаг 5. Прогоняем ВСЕ свои отзывы и считаем.
счёт = {'ПОЗИТИВ':0, 'НЕГАТИВ':0, 'НЕЙТРАЛ':0}
for о in отзывы:
    звёзд, вердикт, увер = разобрать(о)
    счёт[вердикт] += 1
    print(f'{вердикт:8} {звёзд}★  ув.{увер:.0%}  <-  {о}')
print('\nИтог:', счёт)

> ✅ **Проверка.** Каждый отзыв получил вердикт и уверенность, а в конце — словарь с числами.

## Шаг 6 · Хитрые случаи
Сарказм и «не плохо» модель путает. Помечаем спорные — это работа умного инженера.

In [ ]:
# Шаг 6. Ловим ХИТРЫЕ отзывы: низкая уверенность или нейтрал.
print('На проверку человеку:')
for о in отзывы:
    звёзд, вердикт, увер = разобрать(о)
    if увер < 0.6 or вердикт == 'НЕЙТРАЛ':
        print(f'  ⚠ {вердикт} ув.{увер:.0%}  <-  {о}')

> ✅ **Проверка.** В список попали сарказм и отзывы с «не» — там модель менее уверена.

🔶 **Усложни (для сильных):** добавь свой список слов-маркеров сарказма (кавычки, «спасибо, что…») и помечай их отдельно.

## Шаг 7 · Диаграмма
Показываем результат наглядно — столбики позитив/негатив/нейтрал.

In [ ]:
# Шаг 7. Рисуем диаграмму.
import matplotlib.pyplot as plt
метки  = list(счёт.keys())
числа_ = list(счёт.values())
plt.bar(метки, числа_, color=['#2DD4BF','#FF7A8A','#9E96C8'])
plt.title('Радар отзывов: тональность'); plt.ylabel('сколько отзывов'); plt.show()

> ✅ **Проверка.** Появилась столбчатая диаграмма с тремя столбиками.

## Шаг 8 · ⭐ Веб-приложение
Финал: заворачиваем анализатор в сайт с публичной ссылкой (Gradio).

In [ ]:
# Шаг 8 ⭐. Веб-приложение с публичной ссылкой.
!pip install gradio -q
import gradio as gr

def оценить(отзыв):
    звёзд, вердикт, увер = разобрать(отзыв)
    флаг = '  ⚠ стоит проверить человеку' if (увер<0.6 or вердикт=='НЕЙТРАЛ') else ''
    return f'{вердикт} ({звёзд}★, уверенность {увер:.0%}){флаг}'

gr.Interface(fn=оценить, inputs='text', outputs='text',
             title='🛰️ Радар отзывов',
             description='Введи отзыв — модель определит тональность.').launch(share=True)

> ✅ **Проверка.** Появилась ссылка вида https://...gradio.live — открой её на телефоне!

🔶 **Усложни (для сильных):** добавь примеры отзывов кнопками (gr.Interface(..., examples=[...])) и задеплой на Hugging Face Spaces.

## 🏁 Готово! Что у тебя получилось

Ты собрал **работающий продукт**: он читает отзыв и говорит тональность, честно помечает спорные случаи, рисует диаграмму и живёт как сайт по публичной ссылке. Это уже не «ещё один чат-бот» — это твой инструмент.

**Как оценивается мини-проект (10 баллов):**

| Критерий | Баллы |
|---|---|
| Свои данные и постановка (8 отзывов) | 2 |
| Модель подключена и разбирает отзывы | 3 |
| Диаграмма построена | 2 |
| Интерпретация: нашёл, где модель ошиблась, и объяснил почему | 3 |

**Защита (1 минута):** покажи ссылку, введи хитрый отзыв (сарказм) и объясни, почему модель сомневается.